# 02. File I/O, Serialization & Storage Formats (5+ Years Interview Guide)
Exhaustive revision guide to CSV, Fixed-Width, Parquet, Feather, JSON, HTML, and SQL using raw_transactions.csv.

### Key 5-Year Interview Concepts Covered:
- **Text Formats**: Dedicated cell for `pd.read_csv()`, `pd.read_fwf()`, and `df.to_csv()`.
- **Excel Formats**: Dedicated cell for `pd.read_excel()` and `df.to_excel()`.
- **Columnar & Binary Formats**: Dedicated cell for `pd.read_parquet()`, `df.to_parquet()`, and `pd.read_feather()`.
- **Structured Web & DB Formats**: Dedicated cell for `pd.read_json()`, `pd.read_html()`, and `pd.read_sql()`.

This interactive revision guide uses `data/raw_transactions.csv` for all real-world code examples.

In [1]:
# Setup imports & dataset loading
import pandas as pd
import numpy as np
import sys
import time
import os
import sqlite3
import matplotlib.pyplot as plt

# Load raw transactions dataset
csv_path = 'data/raw_transactions.csv' if os.path.exists('data/raw_transactions.csv') else '../data/raw_transactions.csv'
df = pd.read_csv(csv_path)
print(f"Pandas Version: {pd.__version__}")
print(f"Loaded raw_transactions.csv: {df.shape[0]} rows, {df.shape[1]} columns")
print(df.head(2))

Pandas Version: 2.2.2
Loaded raw_transactions.csv: 15000 rows, 11 columns
  transaction_id customer_id merchant_id  ...  transaction_date region is_fraud
0       TX110686      C82845       M2697  ...        07-06-2025  North        0
1       TX107170      C85674       M3868  ...        10/05/2025   East        0

[2 rows x 11 columns]


### Reading CSV with `pd.read_csv()`
**Explanation**: Reads raw_transactions.csv with specified dtypes and parse_dates.

**Syntax**: `pd.read_csv('data/raw_transactions.csv', nrows=1000)`

In [2]:
df_sub = pd.read_csv(csv_path, nrows=10)
print('Read CSV Head (10 rows):\n', df_sub[['transaction_id', 'transaction_amount', 'card_type']])

Read CSV Head (10 rows):
   transaction_id  transaction_amount   card_type
0       TX110686             1216.33        Visa
1       TX107170              324.99  MasterCard
2       TX108328              136.66    Discover
3       TX108563              124.21        Amex
4       TX107002             1284.68  MasterCard
5       TX113784             1263.70        Visa
6       TX111945              781.65        Amex
7       TX103384              363.54        Visa
8       TX107177              679.90        Visa
9       TX112677              559.94  MasterCard


### Reading Fixed-Width Files: `pd.read_fwf()`
**Explanation**: Parses fixed-width banking text feeds into DataFrames.

**Syntax**: `pd.read_fwf(filepath, widths=[...])`

In [3]:
os.makedirs('scratch', exist_ok=True)
with open('scratch/fwf_demo.txt', 'w') as f:
    f.write('TX_1001   500.25    Visa      \nTX_1002   1200.00   MasterCard\n')
df_fwf = pd.read_fwf('scratch/fwf_demo.txt', widths=[10, 10, 12], names=['tx_id', 'amount', 'card'])
print('Parsed Fixed-Width Feed:\n', df_fwf)

Parsed Fixed-Width Feed:
      tx_id   amount        card
0  TX_1001   500.25        Visa
1  TX_1002  1200.00  MasterCard


### Writing CSV with `df.to_csv()`
**Explanation**: Exports filtered transactions to CSV without writing index column.

**Syntax**: `df.to_csv('scratch/clean_tx.csv', index=False)`

In [4]:
df.head(100).to_csv('scratch/sample_tx.csv', index=False)
print('CSV Written. File size:', os.path.getsize('scratch/sample_tx.csv'), 'bytes')

CSV Written. File size: 7717 bytes


### Excel Ingestion & Export: `pd.read_excel()` & `df.to_excel()`
**Explanation**: Spreadsheet format reading and writing via openpyxl.

**Syntax**: `pd.read_excel('file.xlsx')` / `df.to_excel('file.xlsx', index=False)`

In [5]:
print('Excel I/O API: pd.read_excel(filepath, sheet_name=0) / df.to_excel(filepath, index=False)')

Excel I/O API: pd.read_excel(filepath, sheet_name=0) / df.to_excel(filepath, index=False)


### High-Speed Columnar Parquet: `pd.read_parquet()` & `df.to_parquet()`
**Explanation**: Exports raw transactions to Snappy-compressed Parquet and reads with column projection.

**Syntax**: `df.to_parquet('scratch/tx.parquet')` / `pd.read_parquet('scratch/tx.parquet', columns=['transaction_id', 'transaction_amount'])`

In [6]:
df.to_parquet('scratch/raw_transactions.parquet', compression='snappy')
df_proj = pd.read_parquet('scratch/raw_transactions.parquet', columns=['transaction_id', 'transaction_amount', 'is_fraud'])
print('Parquet Column Projection Loaded (5 rows):\n', df_proj.head())

Parquet Column Projection Loaded (5 rows):
   transaction_id  transaction_amount  is_fraud
0       TX110686             1216.33         0
1       TX107170              324.99         0
2       TX108328              136.66         0
3       TX108563              124.21         0
4       TX107002             1284.68         0


### Feather Binary Format: `pd.read_feather()` & `df.to_feather()`
**Explanation**: Ultra-fast inter-process Arrow binary memory format.

**Syntax**: `df.to_feather('scratch/tx.feather')` / `pd.read_feather('scratch/tx.feather')`

In [7]:
df.reset_index(drop=True).to_feather('scratch/raw_transactions.feather')
df_feather = pd.read_feather('scratch/raw_transactions.feather')
print('Feather Loaded Rows:', len(df_feather))

Feather Loaded Rows: 15000


### JSON Parsing: `pd.read_json()` & `df.to_json()`
**Explanation**: Serializes transaction subsets to JSON records and parses them back.

**Syntax**: `df.to_json(orient='records')` / `pd.read_json(json_data)`

In [8]:
json_data = df[['transaction_id', 'transaction_amount', 'card_type']].head(3).to_json(orient='records')
df_json = pd.read_json(json_data, orient='records')
print('Parsed JSON Records:\n', df_json)

<string>:2: FutureWarning: Passing literal json to 'read_json' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
Parsed JSON Records:
   transaction_id  transaction_amount   card_type
0       TX110686             1216.33        Visa
1       TX107170              324.99  MasterCard
2       TX108328              136.66    Discover


### HTML Scraping with `pd.read_html()`
**Explanation**: Scrapes tabular `<table>` HTML elements into DataFrames.

**Syntax**: `pd.read_html(html_str)`

In [9]:
html_table = '<table><tr><th>Region</th><th>Total</th></tr><tr><td>North America</td><td>150000</td></tr></table>'
print('Scraped HTML Table:\n', pd.read_html(html_table)[0])

<string>:2: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
Scraped HTML Table:
           Region   Total
0  North America  150000


### SQL Database Queries with `pd.read_sql()`
**Explanation**: Executes SQL queries against relational databases directly into DataFrames.

**Syntax**: `pd.read_sql('SELECT * FROM transactions WHERE transaction_amount > 500', conn)`

In [10]:
conn = sqlite3.connect(':memory:')
df.head(1000).to_sql('transactions', conn, index=False)
sql_df = pd.read_sql('SELECT card_type, COUNT(*) as tx_count, AVG(transaction_amount) as avg_amt FROM transactions GROUP BY card_type', conn)
print('SQL Aggregated Results:\n', sql_df)

SQL Aggregated Results:
     card_type  tx_count      avg_amt
0        Amex       293  1021.073921
1    Discover       241  1050.927434
2  MasterCard       233   971.707072
3        Visa       233  1006.360811


## Section: Senior Fintech Interview Scenarios (5+ Years Experience)

### Q1: Parquet vs CSV Compression & Predicate Pushdown
**Explanation**: Compare the file size and query speed between `raw_transactions.csv` and `raw_transactions.parquet`.

**Syntax**: `os.path.getsize('file.parquet')` vs `os.path.getsize('file.csv')`

In [11]:
csv_sz = os.path.getsize(csv_path)
pq_sz = os.path.getsize('scratch/raw_transactions.parquet')
print(f'CSV File Size: {csv_sz / 1024:.1f} KB')
print(f'Parquet File Size (Snappy): {pq_sz / 1024:.1f} KB ({(1 - pq_sz/csv_sz)*100:.1f}% reduction)')

CSV File Size: 1106.6 KB
Parquet File Size (Snappy): 345.0 KB (68.8% reduction)
